# Generar modelo SAP2000 (.s2k) desde un IFC

Antes de correr:
1. **Add input → Dataset** con tu archivo `.ifc` (sube el IFC).
2. **Add input → Dataset** con el código: sube `GENERACION-ARCHIVOS-.S2K.zip` (o la carpeta del repositorio).
3. Activa **Internet** en la configuración del notebook (necesario para instalar dependencias).
4. **Run All**. El `.s2k` y el reporte quedan en `/kaggle/working`; descárgalos desde la pestaña *Output*.

El notebook clona siempre la última versión de la rama `claude/compassionate-noether-iwdjcy`. La celda de
resumen muestra las secciones, la grilla y las revisiones de conectividad.

In [ ]:
!pip install -q ifcopenshell shapely trimesh scipy networkx

In [ ]:
import glob
import os
import subprocess
import sys
import zipfile
from pathlib import Path

# ---- Código del generador -------------------------------------------------
# Prioridad: variable S2K_REPO (pruebas) → dataset de Kaggle → clonar desde GitHub.
REPO = None
if os.environ.get("S2K_REPO"):
    REPO = Path(os.environ["S2K_REPO"])
else:
    found = glob.glob("/kaggle/input/**/s2k/__main__.py", recursive=True)
    if found:
        REPO = Path(found[0]).parents[1]
    else:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if zips:
            target = Path("/kaggle/working/src")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(target)
            REPO = Path(glob.glob(str(target / "**/s2k/__main__.py"), recursive=True)[0]).parents[1]
        else:
            REPO = Path("/kaggle/working/GENERACION-ARCHIVOS-.S2K")
            if not REPO.exists():
                subprocess.run(["git", "clone", "--depth", "1", "-b", "claude/compassionate-noether-iwdjcy",
                                "https://github.com/mrubilar84-ship-it/GENERACION-ARCHIVOS-.S2K", str(REPO)], check=True)
print("Código:", REPO)

In [ ]:
# ---- IFC de entrada -------------------------------------------------------
# Si subiste más de un IFC, cambia la línea IFC por la ruta que quieras.
ifcs = sorted(glob.glob("/kaggle/input/**/*.ifc", recursive=True))
if not ifcs:
    raise FileNotFoundError("No encuentro ningún .ifc. Sube el IFC como dataset (Add input).")
print("IFC disponibles:", *ifcs, sep="\n  ")
IFC = ifcs[0]

In [ ]:
# ---- Opciones (valores por defecto del generador) -----------------------
GRID_STEP = 0.10     # paso de la grilla en planta, m (0 = sin grilla)
Z_STEP = 0.05        # paso de los niveles Z, m
GRID_TOL = 0.10      # tolerancia para ajustar a la grilla, m
JOIN = 0.15          # alcance para unir extremos libres, m
SNAP = 0.01          # tolerancia para fusionar nudos, m

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / (Path(IFC).stem + ".s2k")
REPORT = OUT.with_suffix(".report.md")

cmd = [sys.executable, "-m", "s2k", IFC, "-o", str(OUT), "--report", str(REPORT),
       "--grid-step", str(GRID_STEP), "--z-step", str(Z_STEP), "--grid-tol", str(GRID_TOL),
       "--join", str(JOIN), "--snap", str(SNAP)]
res = subprocess.run(cmd, cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO)},
                     capture_output=True, text=True)
print(res.stdout)
if res.returncode not in (0, 1):   # 1 = hay errores de conectividad; igual se generó el archivo
    print(res.stderr)
    raise RuntimeError("El generador falló")
if res.returncode == 1:
    print("Hay errores de conectividad: revisa el reporte antes de usar el modelo.")

In [ ]:
# ---- Resumen: secciones, grilla y revisiones -----------------------------
text = REPORT.read_text(encoding="utf-8")


def section(title):
    parts = text.split("## " + title + "\n", 1)
    if len(parts) < 2:
        return "(sin datos)"
    return parts[1].split("\n## ", 1)[0].strip()


print("SECCIONES\n" + section("Secciones") + "\n")
print("GRILLA\n" + section("Grilla regular") + "\n")
print("REVISIONES\n" + section("Revisiones de conectividad"))

In [ ]:
print(REPORT.read_text(encoding="utf-8"))  # reporte completo

In [ ]:
# Archivos listos para descargar
for f in sorted(OUT_DIR.glob("*")):
    print(f.name, f"{f.stat().st_size/1024:.0f} KB")